# Hướng dẫn Huấn luyện Mô hình YOLOv8x Gộp (Ball, Pitch, Player)

**Lưu ý quan trọng trước khi chạy:**
1. Bật **GPU T4 x2** trong phần Settings của Kaggle (Accelerator -> GPU T4 x2).
2. Đảm bảo bạn đã upload bộ dữ liệu lên Kaggle Dataset và Add Data vào Notebook này.


In [ ]:
# 1. Cài đặt thư viện Ultralytics (YOLO)
%pip install ultralytics==8.2.0 -q
import ultralytics
ultralytics.checks()

In [ ]:
# 2. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 3. Tạo file dataset_merged.yaml trực tiếp
yaml_content = """
path: /kaggle/input/datasets/huynguyen199/dataset-roboflow-for-train-model
train: images/train
val: images/val

names:
  0: ball
  1: pitch
  2: player
"""
with open('/kaggle/working/dataset_merged.yaml', 'w') as f:
    f.write(yaml_content)
print("Đã tạo file dataset_merged.yaml thành công!")

### Bắt đầu Huấn luyện với YOLOv8x (Hỗ trợ Resume & Giới hạn 11 tiếng)

Code dưới đây sẽ sử dụng Python Script thay vì CLI để xử lý logic tự động:
1. Tự động kiểm tra xem có file `last.pt` từ phiên trước không. 
   - Nếu có: Sẽ tự động resume (chạy tiếp từ epoch bị dừng).
   - Nếu không: Sẽ bắt đầu train mới từ đầu.
2. Tham số `time=11.0` giúp giới hạn thời gian train là 11 tiếng. Sau 11 tiếng, mô hình sẽ tự động dừng lại và lưu file `last.pt` một cách an toàn nhất, tránh việc Kaggle ép đóng session sau 12h làm hỏng file model.

In [ ]:
# 4. Khởi chạy quá trình Train
import os
from ultralytics import YOLO

checkpoint_path = '/kaggle/working/Football_Merged/yolov8x_1280p/weights/last.pt'

if os.path.exists(checkpoint_path):
    print("\n--- TÌM THẤY CHECKPOINT TỪ PHIÊN TRƯỚC, BẮT ĐẦU RESUME... ---\n")
    model = YOLO(checkpoint_path)
    model.train(resume=True)
else:
    print("\n--- BẮT ĐẦU TRAIN MỚI TỪ ĐẦU ---\n")
    model = YOLO('yolov8x.pt')
    model.train(
        data='/kaggle/working/dataset_merged.yaml',
        epochs=100,
        imgsz=1280,
        batch=4,
        device=[0,1],
        workers=4,
        project='Football_Merged',
        name='yolov8x_1280p',
        time=11.0 # Giới hạn train trong 11 tiếng
    )

### Kiểm tra kết quả huấn luyện
Dưới đây là các cell để bạn xem đồ thị kết quả sau khi kết thúc 1 phiên train.

In [ ]:
# 5. Xem biểu đồ kết quả Loss/Accuracy sau khi train xong (hoặc ngắt sau 11 tiếng)
from IPython.display import Image
import os

results_dir = '/kaggle/working/Football_Merged/yolov8x_1280p/'
results_img = os.path.join(results_dir, 'results.png')

if os.path.exists(results_img):
    display(Image(filename=results_img))
else:
    print("Chưa có file results.png (có thể mô hình chưa train xong epoch đầu tiên).")

In [ ]:
# 6. Xem ma trận nhầm lẫn (Confusion Matrix)
cm_img = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(cm_img):
    display(Image(filename=cm_img))
else:
    print("Chưa có file confusion matrix.")

In [ ]:
# 7. Chạy kiểm tra Inference thử trên vài ảnh trong tập val
import glob
val_images = glob.glob('/kaggle/input/datasets/huynguyen199/dataset-roboflow-for-train-model/images/val/*.jpg')

if len(val_images) > 0:
    # Lấy 3 ảnh đầu tiên để test
    test_imgs = val_images[:3]
    model_path = '/kaggle/working/Football_Merged/yolov8x_1280p/weights/best.pt'
    
    # Nếu chưa có best.pt thì dùng tạm last.pt
    if not os.path.exists(model_path):
        model_path = '/kaggle/working/Football_Merged/yolov8x_1280p/weights/last.pt'
        
    if os.path.exists(model_path):
        !yolo predict model={model_path} source={test_imgs[0]} imgsz=1280 save=True
    else:
        print("Chưa có model (.pt) nào được lưu để predict thử.")